In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# quick data preprocessing
# aggregating data from both years
df_runners_2019 = pd.read_csv('data/run_ww_2019_q.csv')
df_runners_2020 = pd.read_csv('data/run_ww_2020_q.csv')
df_runners = pd.concat([df_runners_2019, df_runners_2020])
df_runners.reset_index(inplace = True, drop = True)
df_runners['datetime'] = pd.to_datetime(df_runners['datetime'])

df_runners['quarter'] = 'Q' + df_runners['datetime'].dt.quarter.astype(str) + '_' + df_runners['datetime'].dt.year.astype(str)
df_runners.drop(columns = ['country', 'gender', 'number', 'major', 'age_group', 'datetime'], inplace = True)

In [3]:
def center(df, axis = 1, impute = True):
    """Centers DataFrame row(axis=1) or column wise(axis=0) and imputes 0 as centered mean.
    Args:
        df (DataFrame): dataframe containing continous values to center
        axis (int): row(axis=1) or column wise(axis=0); default is 1
        impute (bool): determines whether to impute 0 as the centered mean; default is True
    Returns:
        df (DataFrame): centered and imputed(if applicable) dataframe
    """
    df = df.sub(df.mean(axis = axis), axis = 1 - axis)
    if impute == True:
        df = df.fillna(0)
    return df

In [4]:
def create_pivot(df, value):
    """Creates a pivot table that has athletes as the index and the quartely values as columns
    Args:
        df (DataFrame): data to pivot (consists info taken over 1 year, timestamped to the quarter close)
        value (str): feature column to use for pivot values
    Returns:
        pivot (DataFrame): df of runners quartely info
    """
    pivot = df.pivot_table(index = 'athlete', columns = 'quarter', values = value)
    pivot.columns = [f'{value}_{quarter}' for quarter in pivot.columns]
    return pivot

In [5]:
def runner_collab_similarities(df, target, sim = 'cosine'):
    """Calculates similaty scores for a target runner and all other runners
    
    Args:
        df (DataFrame): pivoted df with runners and quarterly info
        target (int): athlete number to find similarities for
        sim (str): similariy method (cosine or L2); default is cosine
        
    Returns:
        similarities (Series): pd series of similarity scores between target runners and other runners
    """  
    df_copy = center(df.copy(), axis = 1, impute = True)
    
    # pairwise sim scores (only for target runner)
    if sim == 'cosine':
        similarity_matrix = cosine_similarity(df_copy.loc[[target]], df_copy)
        similarities = pd.Series(similarity_matrix[0], index = df.index)
    elif sim == 'L2':
        euclid_distances = -np.sqrt(((df_copy.loc[[target]].values - df_copy.values) ** 2).sum(axis = 1))
        similarities = pd.Series(euclid_distances, index = df.index)
        
    # exclude target runner's self similarity
    similarities[target] = np.nan
    
    # min-max scaling
    min_val = similarities.min()
    max_val = similarities.max()
    similarities = (similarities - min_val) / (max_val - min_val)
    
    return similarities


def runner_collab_filter(df, target, features=['distance', 'duration'], sim='cosine', k=3, predict_missing=True):
    """Collaborative filtering to find most similar runners or predict missing features, based on the averaged similarity
    scores of all features
    Args:
        df (DataFrame): runner data with an entry per runner per quarter
        target (int): target runner to find most similar runners
        features (list): list of features to use to determine similarity
        sim (str): similariy method (cosine or L2); default is cosine
        k (int): number of neighbors to find
        predict_missing (bool): optional to fill any missing values based on k most similar runners
        
    Returns:
        (dict): dictionary containing k closest runners 'running_buddies' and their corresponding simialrity 
        scores 'similarities', and the missing predictions if any 'predictions'
    """
    # compute similarities for each feature and average all similarities
    all_similarities = []
    pivots = {}
    # creating pivots and finding similarites for all features
    for feature in features:
        pivot = create_pivot(df, feature)
        pivots[feature] = pivot
        
        # ensuring target runner exists in df
        if target not in pivot.index:
            print(f"Error: {target} not found in {feature} pivot.")
            return None
        
        similarities = runner_collab_similarities(pivot, target, sim = sim)
        all_similarities.append(similarities)
    
    # finding most similar runners on average (based on averaged sim scores across all targeted features)
    avg_similarities = pd.concat(all_similarities, axis = 1).mean(axis = 1)
    avg_similarities[target] = np.nan

    # find k closest runners
    k_closest = avg_similarities.nlargest(k).index
    weights = avg_similarities[k_closest]

    # predict missing values per feature using original pivot
    predictions = {}
    if predict_missing:
        for feature, pivot in pivots.items():
            missing_cols = [col for col in pivot.columns if pd.isna(pivot.loc[target, col])]
            if missing_cols:
                neighbor_vals = pivot.loc[k_closest, missing_cols].copy()
                neighbor_vals = neighbor_vals.fillna(neighbor_vals.mean())
                predictions[feature] = (neighbor_vals.T.dot(weights) / weights.sum()).to_dict()

    return {
        'running_buddies': k_closest,
        'similarities': avg_similarities[k_closest].round(4),
        'predictions': predictions
    }

In [6]:
running_match = runner_collab_filter(df_runners, 1)
running_match

{'running_buddies': Index([2163, 33818, 9070], dtype='int64', name='athlete'),
 'similarities': athlete
 2163     0.9997
 33818    0.9975
 9070     0.9969
 dtype: float64,
 'predictions': {}}

In [7]:
distance_pivot = create_pivot(df_runners, 'distance')
duration_pivot = create_pivot(df_runners, 'duration')

In [8]:
print('Target User 1')
print('Distances:')
print(distance_pivot.loc[1])
print()
print('Durations:')
print(duration_pivot.loc[1])
print()
print(f'Average Distance: {round(distance_pivot.loc[1].mean(), 3)}')
print(f'Average Duration: {round(duration_pivot.loc[1].mean(), 3)}')

Target User 1
Distances:
distance_Q1_2019    718.350
distance_Q1_2020    689.548
distance_Q2_2019    661.560
distance_Q2_2020    506.119
distance_Q3_2019    757.100
distance_Q3_2020    417.150
distance_Q4_2019    633.878
distance_Q4_2020    516.458
Name: 1, dtype: float64

Durations:
duration_Q1_2019    4064.200000
duration_Q1_2020    4056.516667
duration_Q2_2019    3901.133333
duration_Q2_2020    2933.266667
duration_Q3_2019    4352.083333
duration_Q3_2020    2417.016667
duration_Q4_2019    3664.316667
duration_Q4_2020    2928.900000
Name: 1, dtype: float64

Average Distance: 612.52
Average Duration: 3539.679


In [9]:
for runner, sim_score in running_match['similarities'].items():
    athlete = df_runners.loc[runner]['athlete']
    
    print(f'Similar Runner: {athlete}')
    print(f'Similarity Score: {sim_score}')
    print()
    print('Distances:')
    print(distance_pivot.loc[athlete])
    print()
    print('Durations:')
    print(duration_pivot.loc[athlete])
    print()
    print(f'Average Distance: {round(distance_pivot.loc[athlete].mean(), 3)}')
    print(f'Average Duration: {round(duration_pivot.loc[athlete].mean(), 3)}')
    print()

Similar Runner: 2269
Similarity Score: 0.9997

Distances:
distance_Q1_2019    495.979
distance_Q1_2020    458.940
distance_Q2_2019    446.600
distance_Q2_2020    398.469
distance_Q3_2019    719.640
distance_Q3_2020    375.509
distance_Q4_2019    295.960
distance_Q4_2020    258.278
Name: 2269, dtype: float64

Durations:
duration_Q1_2019    3175.016667
duration_Q1_2020    2881.550000
duration_Q2_2019    2796.366667
duration_Q2_2020    2312.616667
duration_Q3_2019    4381.550000
duration_Q3_2020    2816.083333
duration_Q4_2019    1880.850000
duration_Q4_2020    1604.133333
Name: 2269, dtype: float64

Average Distance: 431.172
Average Duration: 2731.021

Similar Runner: 34884
Similarity Score: 0.9975

Distances:
distance_Q1_2019     341.459
distance_Q1_2020     969.287
distance_Q2_2019    1025.419
distance_Q2_2020     950.650
distance_Q3_2019     989.130
distance_Q3_2020    1146.059
distance_Q4_2019     630.389
distance_Q4_2020    1481.580
Name: 34884, dtype: float64

Durations:
duration_Q